# 04. Model Training
Train multiple recommendation models.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

In [ ]:
events = pd.read_csv('../data/user_events.csv')
products = pd.read_csv('../data/products.csv')

## 1. Popularity Based (Trending)

In [ ]:
# Simple popularity model based on view/purchase counts
product_popularity = events.groupby('product_id').size().reset_index(name='score')
product_popularity = product_popularity.sort_values('score', ascending=False)
product_popularity.to_csv('../models/popularity_model.csv', index=False)

## 2. Collaborative Filtering (Item-Item)

In [ ]:
# Filter to subset of users/items for matrix to avoid memory issues locally
active_users = events['user_id'].value_counts()[events['user_id'].value_counts() > 5].index
events_filtered = events[events['user_id'].isin(active_users)].copy()


In [ ]:
user_item_matrix = events_filtered.pivot_table(index='user_id', columns='product_id', values='event_type', aggfunc='count', fill_value=0)
sparse_matrix = csr_matrix(user_item_matrix.values)
item_similarity = cosine_similarity(sparse_matrix.T)
np.save('../models/item_similarity.npy', item_similarity)
with open('../models/user_item_matrix_cols.pkl', 'wb') as f:
    pickle.dump(user_item_matrix.columns, f)
with open('../models/user_item_matrix_idx.pkl', 'wb') as f:
    pickle.dump(user_item_matrix.index, f)